# Chapter 3 / Paper 2

## Notebook 2: GEWI Construction

This notebook documents the preparation of geolocated digital discourse, sentiment scoring, message-volume adjustment, tract-level GEWI construction, polarity decomposition, and spatial coverage procedures.

> **Public analytical copy.** Raw geolocated messages and restricted derived files are not included. Run the notebook from the `chapter_3_paper_2` directory with authorized inputs described in `data/README.md`.

In [ ]:
from pathlib import Path

CHAPTER_DIR = Path.cwd().resolve()
if CHAPTER_DIR.name == 'notebooks':
    CHAPTER_DIR = CHAPTER_DIR.parent
if not (CHAPTER_DIR / 'data').exists():
    raise RuntimeError(
        'Start Jupyter from the chapter_3_paper_2 directory or its notebooks directory.'
    )

# Original analytical code below uses paths relative to the chapter directory.
import os
os.chdir(CHAPTER_DIR)


In [ ]:
##  Block 1 — Vocabulary + Regex

In [ ]:
import re
import unidecode

# Final list of validated keywords and expressions
keywords = [
    #  Outlets and switches
    "tomada", "tomadas", "interruptor", "interruptores",
    "tomada queimada", "tomada não funciona", "trocar tomada", "colocar tomada", "tomada derretida",

    #  Circuit breakers and panels
    "disjuntor", "minidisjuntor", "quadro de luz", "painel elétrico", "caixa de disjuntores",
    "curto-circuito", "curto circuito", "disjuntor caiu", "caiu a luz",

    #  Wiring and cables
    "fiação", "fio elétrico", "fios e cabos", "cabo elétrico", "fio solto", "fio desencapado",
    "ligação elétrica", "trocando os fios", "fio exposto", "ligar os fios",

    #  Conduits and cable ducts
    "eletroduto", "canaleta", "tubo para fiação", "cano elétrico", "tubo corrugado", "passagem de fios",

    #  Electrical boxes and condulets
    "caixa de luz", "caixa de disjuntores", "condulete", "caixa elétrica", "caixa de energia",

    #  Insulating tapes and cable ties
    "fita isolante", "fita preta", "abraçadeira de nylon", "enforca gato", "presilha de fio",

    #  Power strips and plugs
    "extensão elétrica", "benjamin", "filtro de linha", "régua elétrica", "tomada múltipla",
    "plug", "adaptador", "pino redondo",

    #  Sensors and cable organizers
    "sensor de presença", "luz automática", "organizador de fio", "organizar cabos", "fio enrolado",

    #  Terminals and sleeves
    "terminal elétrico", "luva de compressão", "emenda de fio", "encaixe de fio",

    #  Renovation and construction
    "reforma elétrica", "obra elétrica", "obra em casa", "construção", "obra residencial", "reforma geral",

    #  Installation and maintenance
    "instalação elétrica", "instalar tomada", "instalando fios", "manutenção elétrica", "serviço elétrico",

    #  Professionals
    "eletricista", "chamei o eletricista", "veio o eletricista", "técnico de elétrica",

    #  Common problems
    "tomada com mau contato", "choque elétrico", "queda de energia", "energia oscilando",
    "problema na fiação", "tomada estalando"
]

# Compile regex with normalized keywords
def prepare_regex(words):
    normalized_terms = [re.escape(unidecode.unidecode(w.lower())) for w in words]
    return r'\b(?:' + '|'.join(normalized_terms) + r')\b'

compiled_regex = re.compile(prepare_regex(keywords), flags=re.IGNORECASE)

print("Regex compiled successfully.")

In [ ]:
## Block 2 — RAN VIA TERMINAL — output_base = "data/private/gewi/"

In [ ]:
## DO NOT RUN
## EXECUTED VIA TERMINAL
# ========== VALIDATED KEYWORD BLOCKS ==========
keywords = [
    # Outlets and switches
    "tomada", "tomadas", "interruptor", "interruptores",
    "tomada queimada", "tomada não funciona", "trocar tomada", "colocar tomada", "tomada derretida",

    # Circuit breakers and panels
    "disjuntor", "minidisjuntor", "quadro de luz", "painel elétrico", "caixa de disjuntores",
    "curto-circuito", "curto circuito", "disjuntor caiu", "caiu a luz",

    # Wiring and cables
    "fiação", "fio elétrico", "fios e cabos", "cabo elétrico", "fio solto", "fio desencapado",
    "ligação elétrica", "trocando os fios", "fio exposto", "ligar os fios",

    # Conduits and cable ducts
    "eletroduto", "canaleta", "tubo para fiação", "cano elétrico", "tubo corrugado", "passagem de fios",

    # Electrical boxes and condulets
    "caixa de luz", "caixa de disjuntores", "condulete", "caixa elétrica", "caixa de energia",

    # Insulating tapes and cable ties
    "fita isolante", "fita preta", "abraçadeira de nylon", "enforca gato", "presilha de fio",

    # Power strips and plugs
    "extensão elétrica", "benjamin", "filtro de linha", "régua elétrica", "tomada múltipla",
    "plug", "adaptador", "pino redondo",

    # Sensors and organizers
    "sensor de presença", "luz automática", "organizador de fio", "organizar cabos", "fio enrolado",

    # Terminals and sleeves
    "terminal elétrico", "luva de compressão", "emenda de fio", "encaixe de fio",

    # Renovation and construction
    "reforma elétrica", "obra elétrica", "obra em casa", "construção", "obra residencial", "reforma geral",

    # Installation and maintenance
    "instalação elétrica", "instalar tomada", "instalando fios", "manutenção elétrica", "serviço elétrico",

    # Professionals
    "eletricista", "chamei o eletricista", "veio o eletricista", "técnico de elétrica",

    # Common problems
    "tomada com mau contato", "choque elétrico", "queda de energia", "energia oscilando",
    "problema na fiação", "tomada estalando"
]

def prepare_regex(words):
    normalized_terms = [re.escape(unidecode.unidecode(w.lower())) for w in words]
    return r'\b(?:' + '|'.join(normalized_terms) + r')\b'

compiled_regex = re.compile(prepare_regex(keywords), flags=re.IGNORECASE)

# ========== PATH CONFIGURATION ==========
input_base = "data/private/geolocated_discourse/"
output_base = "data/private/gewi/"
os.makedirs(output_base, exist_ok=True)

num_workers = 64

# ========== FILE PROCESSING FUNCTION ==========
def process_file(file_path):
    try:
        df = pd.read_csv(
            file_path,
            sep="\t",
            compression="gzip",
            engine="python",
            on_bad_lines="skip"
        )

        if "text" not in df.columns or "latitude" not in df.columns or "longitude" not in df.columns:
            return pd.DataFrame()

        df = df.dropna(subset=["text", "latitude", "longitude"])
        df = df[df["tweet_lang"] == "pt"]
        df["text_norm"] = df["text"].astype(str).apply(lambda x: unidecode.unidecode(x.lower()))
        df_filtered = df[df["text_norm"].str.contains(compiled_regex)]
        return df_filtered.drop(columns=["text_norm"])

    except Exception as e:
        print(f" Error in {os.path.basename(file_path)}: {e}", flush=True)
        return pd.DataFrame()

# ========== LIST OF YEAR-MONTH PAIRS ==========
year_month_pairs = [
    (year, month) for year in range(2010, 2024)
    for month in range(1, 13)
    if not (year == 2023 and month > 7)
]

# ========== MAIN LOOP ==========
for year, month in tqdm(year_month_pairs, desc="Processing months"):
    print(f"\n Starting {year}/{month:02d}...\n", flush=True)

    input_dir = os.path.join(input_base, str(year))
    output_file = os.path.join(output_base, f"sector_relevant_messages_{year}_{month:02d}.csv.gz")

    try:
        month_files = sorted([
            os.path.join(input_dir, f) for f in os.listdir(input_dir)
            if f.startswith(f"{year}_{month}_") and f.endswith(".csv.gz")
        ])
    except FileNotFoundError:
        print(f" {year}/{month:02d} — Directory not found", flush=True)
        continue

    if not month_files:
        print(f" {year}/{month:02d} — No files found", flush=True)
        continue

    with Pool(processes=num_workers) as pool:
        results = list(pool.map(process_file, month_files))

    results = [df for df in results if not df.empty]

    if results:
        df_final = pd.concat(results, ignore_index=True)
        df_final.to_csv(output_file, index=False, compression="gzip")
        print(f" {year}/{month:02d} — {len(df_final)} tweets saved → {output_file}", flush=True)
    else:
        print(f" {year}/{month:02d} — No valid tweets found", flush=True)

    del results

In [ ]:
##  Block 3 — Semantic Validation with BERT + Cosine Similarity

In [ ]:
##  Block 3 — Semantic Validation with BERT + Cosine Similarity

import sys

import os
import pandas as pd
import numpy as np
from tqdm import tqdm
from sentence_transformers import SentenceTransformer, util

# Paths
input_base = "data/private/gewi/"
output_base = "data/private/gewi/validados/"
os.makedirs(output_base, exist_ok=True)

#  Load BERT model optimized for multilingual use
print(" Loading BERT model...", flush=True)
bert_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

#  Anchor phrases that represent real e-WOM cases
anchor_phrases = [
    "comprei um disjuntor",
    "estou trocando a fiação elétrica da casa",
    "vou instalar novas tomadas e interruptores",
    "contratei um eletricista para refazer a instalação",
    "preciso de um novo quadro de luz",
    "o fio queimou e tive que trocar",
    "tive curto-circuito em casa",
    "preciso de cabo elétrico novo",
    "o eletricista recomendou trocar a caixa de disjuntores",
    "estou reformando a parte elétrica do apartamento"
]
print(" Generating embeddings for anchor phrases...", flush=True)
anchor_embeddings = bert_model.encode(anchor_phrases, convert_to_tensor=True)

# Analysis period: Jan/2010 to Jul/2023
year_month_pairs = [
    (year, month) for year in range(2010, 2024)
    for month in range(1, 13)
    if not (year == 2023 and month > 7)
]

similarity_threshold = 0.55  # Adjustable for more strict filtering

# Main loop: month by month
for year, month in tqdm(year_month_pairs, desc="Validating months with BERT"):
    filename = f"sector_relevant_messages_{year}_{month:02d}.csv.gz"
    input_path = os.path.join(input_base, filename)
    output_path = os.path.join(output_base, filename)

    print(f"\n Starting {year}/{month:02d}...", flush=True)

    if not os.path.exists(input_path):
        print(f" File not found: {input_path}", flush=True)
        continue

    try:
        df = pd.read_csv(input_path)
        df = df.dropna(subset=["text"])
        texts = df["text"].astype(str).tolist()
        print(f" {len(texts)} tweets found for the month", flush=True)

        # Generate tweet embeddings
        print(" Generating embeddings with BERT...", flush=True)
        tweet_embeddings = bert_model.encode(texts, convert_to_tensor=True, batch_size=128, show_progress_bar=True)

        # Calculate similarity with anchor phrases
        print(" Calculating semantic similarities...", flush=True)
        similarities = util.cos_sim(tweet_embeddings, anchor_embeddings).max(dim=1).values
        df["similarity"] = similarities.cpu().numpy()

        # Filter tweets above similarity threshold
        df_filtered = df[df["similarity"] >= similarity_threshold]
        print(f" {len(df_filtered)} relevant tweets saved → {output_path}", flush=True)

        df_filtered.to_csv(output_path, index=False, compression="gzip")

        del df, df_filtered, tweet_embeddings, similarities
    except Exception as e:
        print(f" Error processing {year}/{month:02d} — {e}", flush=True)

In [ ]:
## Block 3 Verification

In [ ]:
import os
import pandas as pd
from tqdm import tqdm

# Path to the folder containing validated tweet files
validated_path = "data/private/gewi/validados/"

# List all CSV.GZ files in the folder
files = sorted([
    f for f in os.listdir(validated_path)
    if f.endswith(".csv.gz")
])

total_tweets = 0
tweets_per_month = []

for file in tqdm(files, desc="Counting validated tweets"):
    file_path = os.path.join(validated_path, file)
    try:
        df = pd.read_csv(file_path, usecols=["text"])  # Lightweight read
        n = len(df)
        total_tweets += n
        tweets_per_month.append((file, n))
    except Exception as e:
        print(f"Error reading {file}: {e}")

# Print total and top 10 months by tweet volume
print(f"\n Total validated tweets (BERT): {total_tweets:,}")

print("\n Top 10 months with the most tweets:")
for name, n in sorted(tweets_per_month, key=lambda x: x[1], reverse=True)[:10]:
    print(f"{name}: {n:,} tweets")

In [ ]:
##  Block 4 — Sample-Based Qualitative Validation

In [ ]:
import os
import pandas as pd
from tqdm import tqdm

# Updated paths
input_base = "data/private/gewi/validados/"
output_samples = "data/private/gewi/amostras_qualitativas/"
os.makedirs(output_samples, exist_ok=True)

# Full period of the validated dataset
year_month_pairs = [
    (year, month) for year in range(2010, 2024)
    for month in range(1, 13)
    if not (year == 2023 and month > 7)
]

# Number of tweets per sample
sample_size = 20

# Main loop
for year, month in tqdm(year_month_pairs, desc="Generating qualitative samples"):
    filename = f"sector_relevant_messages_{year}_{month:02d}.csv.gz"
    input_path = os.path.join(input_base, filename)
    output_path = os.path.join(output_samples, f"sample_{year}_{month:02d}.csv")

    if not os.path.exists(input_path):
        print(f" {year}/{month:02d} — File not found")
        continue

    try:
        df = pd.read_csv(input_path)
        if df.empty:
            print(f" {year}/{month:02d} — Empty file")
            continue

        sample = df.sample(n=min(sample_size, len(df)), random_state=42)[["date", "text", "similarity"]]
        sample.to_csv(output_path, index=False)
        print(f" {year}/{month:02d} — {len(sample)} tweets saved to {output_path}")
    except Exception as e:
        print(f" Error in {year}/{month:02d} — {e}")

In [ ]:
## Block 4b —  Code to Load and Visualize Qualitative Tweet Samples

In [ ]:
import pandas as pd
import os

# Updated path for qualitative samples
samples_path = "data/private/gewi/amostras_qualitativas/"

# List available sample files
print(" Available files:")
print(sorted(os.listdir(samples_path)))

#  File name to visualize
file_name = "sample_2020_01.csv"

# Full path
file_path = os.path.join(samples_path, file_name)

# Read and display sample
df = pd.read_csv(file_path)
print("\n Sample of tweets:")
print(df.head(20))

In [ ]:
##  Block 5 — Initial Exploratory Analysis

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

# Path to validated tweets
validated_path = "data/private/gewi/validados/"
files = sorted([f for f in os.listdir(validated_path) if f.endswith(".csv.gz")])

# Aggregate data by month
data = []
for file in files:
    try:
        df = pd.read_csv(os.path.join(validated_path, file))
        year_month = file.replace("sector_relevant_messages_", "").replace(".csv.gz", "")
        data.append({
            "year_month": year_month,
            "n_tweets": len(df),
            "mean_similarity": df["similarity"].mean(),
            "std_similarity": df["similarity"].std()
        })
    except Exception as e:
        print(f" Error processing {file}: {e}", flush=True)
        continue

# Create summary DataFrame
df_summary = pd.DataFrame(data).sort_values("year_month")

#  Volume of validated tweets per month
plt.figure(figsize=(14, 5))
plt.plot(df_summary["year_month"], df_summary["n_tweets"], marker="o", linewidth=2)
plt.xticks(rotation=45)
plt.title(" Volume of Validated Tweets per Month (e-WOM)")
plt.ylabel("Number of Tweets")
plt.grid(True)
plt.tight_layout()
plt.show()

#  Average semantic similarity per month
plt.figure(figsize=(14, 5))
plt.plot(df_summary["year_month"], df_summary["mean_similarity"], marker="s", color="green", linewidth=2)
plt.xticks(rotation=45)
plt.title(" Average Semantic Similarity per Month")
plt.ylabel("Similarity (cosine)")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
import os
import pandas as pd
import random

# Path to the folder containing validated tweet files
validated_folder = "data/private/gewi/validados/"

# List all available .csv.gz files
files = [f for f in os.listdir(validated_folder) if f.endswith(".csv.gz")]

# Select a random file
random_file = random.choice(files)
print(f" Selected file: {random_file}")

# Load only the header to preview columns
file_path = os.path.join(validated_folder, random_file)
df = pd.read_csv(file_path, nrows=5)  # Load only the first few rows
print("\n Available columns:")
print(df.columns.tolist())

In [ ]:
##   Block 6 — Custom Sentiment Analysis (Hugging Face + GPU)

In [ ]:
import sys


In [ ]:
#  Block 6 — Sentiment Analysis with Hugging Face (final version)
import os
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
import torch
from tqdm import tqdm

# Input and output directories
input_dir = "data/private/gewi/validados/"
output_dir = "data/private/gewi/validados_com_sentimento_v4/"
os.makedirs(output_dir, exist_ok=True)

# Detect GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f" Using device: {device}")

# Load sentiment model from Hugging Face
MODEL_NAME = "nlptown/bert-base-multilingual-uncased-sentiment"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).to(device)
sentiment_fn = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer, device=0 if device == "cuda" else -1)

# Robust sentiment analysis function
def analyze_sentiment(text):
    try:
        result = sentiment_fn(text[:512])[0]
        return result.get("score", None)
    except:
        return None

# Generate list of files (Jan 2010 to Jul 2023)
year_month_pairs = [
    (year, month) for year in range(2010, 2024)
    for month in range(1, 13)
    if not (year == 2023 and month > 7)
]

# Process each file
for year, month in tqdm(year_month_pairs, desc=" Analyzing sentiment per month"):
    file_name = f"sector_relevant_messages_{year}_{month:02d}.csv.gz"
    path_in = os.path.join(input_dir, file_name)
    path_out = os.path.join(output_dir, file_name)

    if not os.path.exists(path_in):
        print(f" File not found: {file_name}")
        continue

    try:
        df = pd.read_csv(path_in)
        if "text" not in df.columns or df.empty:
            print(f" {file_name} skipped (missing 'text' column or empty file)")
            continue

        print(f" Processing {file_name} — {len(df)} tweets")
        df["score"] = df["text"].astype(str).apply(analyze_sentiment)
        df.to_csv(path_out, index=False, compression="gzip")
        print(f" Sentiment scores saved to: {path_out}\n")

    except Exception as e:
        print(f" Error processing {file_name}: {e}")

In [ ]:
## Block 6 Verification

In [ ]:
import pandas as pd
import os

#  Updated path to tweets with sentiment scores (v4)
file_path = "data/private/gewi/validados_com_sentimento_v4/sector_relevant_messages_2020_01.csv.gz"

# Check if the file exists
if os.path.exists(file_path):
    df = pd.read_csv(file_path, compression="gzip")
    print(f" {len(df)} tweets found in {file_path}")
    print(df.head())
else:
    print(f" File not found: {file_path}")

In [ ]:
##  Block 7 — Geographic Mapping by Municipality

In [ ]:
##  Block 7 — Geographic Mapping by Municipality (updated version)

import os
import gzip
import json
import requests
import pandas as pd
from tqdm import tqdm
from shapely.geometry import Point, shape

# Updated directories
input_dir = "data/private/gewi/validados_com_sentimento_v4/"
output_dir = "data/private/gewi/mapped_municipalities_v1/"
os.makedirs(output_dir, exist_ok=True)

#  Simplified GeoJSON of Brazilian municipalities
url_geojson = "https://raw.githubusercontent.com/dadosfera/brasil-municipios-geojson/refs/heads/main/geojs-100-mun-v2.json"
print(" Downloading simplified municipality shapefile...")
geojson_data = requests.get(url_geojson).json()

#  Create list of polygons with municipality names
municipalities = []
for feature in geojson_data["features"]:
    name = feature["properties"]["name"]
    polygon = shape(feature["geometry"])
    municipalities.append((name, polygon))

#  Files to process
files = sorted([f for f in os.listdir(input_dir) if f.endswith(".csv.gz")])
print(f" {len(files)} files found for geographic mapping.\n")

#  Mapping loop
for file in tqdm(files, desc="Mapping tweets by municipality"):
    path_in = os.path.join(input_dir, file)
    path_out = os.path.join(output_dir, file)

    try:
        df = pd.read_csv(path_in, compression="gzip")

        if "latitude" not in df.columns or "longitude" not in df.columns:
            print(f" Coordinate columns missing in: {file}")
            continue

        #  Location function
        def locate_municipality(row):
            point = Point(row["longitude"], row["latitude"])
            for name, polygon in municipalities:
                if polygon.contains(point):
                    return name
            return "Undefined"

        df["municipality"] = df.apply(locate_municipality, axis=1)

        df.to_csv(path_out, index=False, compression="gzip")
        print(f" Mapped and saved: {file}")

    except Exception as e:
        print(f" Error processing {file} — {e}")

In [ ]:
## Block 7 Verification

In [ ]:
##  Verification by City (updated version)

import os
import random
import pandas as pd

# Path to files mapped with sentiment and municipality
mapped_dir = "data/private/gewi/mapped_municipalities_v1/"

# List available files
files = [f for f in os.listdir(mapped_dir) if f.endswith(".csv.gz")]

# Select a random file
random_file = random.choice(files)
print(f" Checking random file: {random_file}\n")

# Load the selected file
df = pd.read_csv(os.path.join(mapped_dir, random_file), compression="gzip")

# Display the first rows with the 'municipality' column
print(df.head(10))

In [ ]:
import os
import pandas as pd
from tqdm import tqdm

# Path to the folder with municipality-mapped tweet files
input_dir = "data/private/gewi/mapped_municipalities_v1/"

# List of compressed CSV files
files = sorted([f for f in os.listdir(input_dir) if f.endswith(".csv.gz")])

# Counter for total number of rows
total_rows = 0

print(" Counting total rows (tweets) in municipality-mapped files...\n")
for file in tqdm(files):
    file_path = os.path.join(input_dir, file)
    try:
        df = pd.read_csv(file_path, usecols=["message_id"])  # Minimal read for performance
        total_rows += len(df)
    except Exception as e:
        print(f" Error processing {file}: {e}")

print(f"\n Total number of rows (tweets) in municipality-mapped files: {total_rows:,}")

In [ ]:
##  Block 8 — Mapping by Census Tract — EXECUTED VIA TERMINAL FOR PERFORMANCE

In [ ]:
#  DO NOT RUN — Block 8 — Mapping by Census Tract with Multiprocessing

import os
import json
import pandas as pd
from shapely.geometry import Point, shape
from tqdm import tqdm
from multiprocessing import Pool, cpu_count

print(f" Detected CPU cores: {cpu_count()}")

# Paths
input_dir = "data/private/gewi/validados_com_sentimento_v4/"
output_dir = "data/private/gewi/mapped_census_tracts_v2/"
geojson_path = "data/private/census_tracts/municipios_simplificados.geojson"

# Load GeoJSON with census tracts
print(" Reading census tract GeoJSON...")
with open(geojson_path, 'r', encoding='utf-8') as f:
    census_data = json.load(f)

# Build tract dictionary
tracts = {}
for feature in census_data['features']:
    tract_id = feature['properties'].get('CD_SETOR')
    geometry = shape(feature['geometry'])
    tracts[tract_id] = geometry

print(f" Total census tracts loaded: {len(tracts)}\n")

# Helper function to map a single tweet
def map_one_tweet(tweet):
    try:
        point = Point(tweet["longitude"], tweet["latitude"])
        for tract_id, polygon in tracts.items():
            if polygon.contains(point):
                return tract_id
        return None
    except:
        return None

# Function to process one file
def process_file(filename):
    path_in = os.path.join(input_dir, filename)
    path_out = os.path.join(output_dir, filename)

    try:
        df = pd.read_csv(path_in)
        if "latitude" not in df.columns or "longitude" not in df.columns:
            print(f" Coordinates missing in {filename}")
            return

        points = df[["longitude", "latitude"]].to_dict(orient="records")

        with Pool(processes=cpu_count()) as pool:
            mapped_tracts = list(pool.map(map_one_tweet, points))

        df["CD_SETOR"] = mapped_tracts
        df["mapped_tract"] = df["CD_SETOR"].notna()
        df.to_csv(path_out, index=False, compression="gzip")

        print(f" Mapped and saved: {filename}")
    except Exception as e:
        print(f" Error in {filename}: {e}")

# Create output directory
os.makedirs(output_dir, exist_ok=True)

# Run all files (each one is processed in parallel internally)
files = sorted([f for f in os.listdir(input_dir) if f.endswith(".csv.gz")])
print(f" {len(files)} files found for census tract mapping.\n")

for filename in tqdm(files, desc="Mapping tweets by census tract"):
    process_file(filename)

In [ ]:
import os
import pandas as pd
import random

# Path to census tract-mapped files
output_dir = "data/private/gewi/mapeado_setores_v2/"

# List only files from 2023
arquivos_2023 = sorted([f for f in os.listdir(output_dir) if f.startswith("sector_relevant_messages_2023") and f.endswith(".csv.gz")])

# Select a random file from 2023
arquivo_escolhido = random.choice(arquivos_2023)
print(f"\n Verificando mês aleatório de 2023: {arquivo_escolhido}")  #  Mantido em português

# Read the file
df = pd.read_csv(os.path.join(output_dir, arquivo_escolhido), compression="gzip")

# Display the first rows with census tract info
print(df[["text", "latitude", "longitude", "CD_SETOR"]].head(10))  #  Mantido como está

In [ ]:
## Output Verification — Block 8

In [ ]:
import pandas as pd
import os

# Path to the folder with census tract-mapped files
output_dir = "data/private/gewi/mapeado_setores_v2/"

# Listing the files in the folder
arquivos = [f for f in os.listdir(output_dir) if f.endswith(".csv.gz")]

# Inspecting the first few files
for arq in arquivos[:5]:  # You can adjust the number of files to inspect
    path = os.path.join(output_dir, arq)
    df = pd.read_csv(path, compression='gzip')
    print(f"Primeiras linhas do arquivo {arq}:\n", df.head(), "\n")  #  Mantido em português

In [ ]:
import os
import pandas as pd
from tqdm import tqdm

# Path to the folder with census tract-mapped files
input_dir = "data/private/gewi/mapeado_setores_v2/"

# List of compressed CSV files
files = sorted([f for f in os.listdir(input_dir) if f.endswith(".csv.gz")])

# Counter for total number of rows
total_rows = 0

print(" Counting total number of rows (tweets) in census tract-mapped files...\n")
for file in tqdm(files):
    file_path = os.path.join(input_dir, file)
    try:
        df = pd.read_csv(file_path, usecols=["message_id"])  # Minimal read for performance
        total_rows += len(df)
    except Exception as e:
        print(f" Error processing {file}: {e}")

print(f"\n Total number of rows (tweets) in census tract-mapped files: {total_rows:,}")

In [ ]:
##  Block 9 — Calculation of GEWI Indices (v1, v2, v3)

In [ ]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

# Path to tweets mapped by census tract
input_dir = "data/private/gewi/mapeado_setores_v2/"
output_path = "data/private/gewi/gewis_by_tract.csv"

# List of .csv.gz files
files = sorted([f for f in os.listdir(input_dir) if f.endswith(".csv.gz")])

# List to store results
results = []

print(" Calculating GEWI v1, v2, and v3 by census tract...\n")

for file in tqdm(files, desc="Processing files"):
    df = pd.read_csv(os.path.join(input_dir, file), compression="gzip")

    if "CD_SETOR" not in df.columns or df.empty:
        continue

    # Create auxiliary columns
    df["followers_log"] = np.log1p(df["followers"])
    df["year_month"] = file.replace("sector_relevant_messages_", "").replace(".csv.gz", "")

    # Grouping by census tract
    for tract_id, group in df.groupby("CD_SETOR"):
        n_total = len(group)
        mean_sentiment = group["score"].mean()
        log_volume = np.log1p(n_total)
        gewi_v1 = mean_sentiment * log_volume

        # v2 - positive and negative polarity
        positives = group[group["score"] >= 0.5]
        negatives = group[group["score"] < 0.5]

        n_pos = len(positives)
        n_neg = len(negatives)

        gewi_pos = positives["score"].mean() * np.log1p(n_pos) if n_pos > 0 else 0
        gewi_neg = negatives["score"].mean() * np.log1p(n_neg) if n_neg > 0 else 0

        # v3 - weighted by log(followers)
        weights = group["followers_log"].fillna(0)
        weighted_avg = np.average(group["score"], weights=weights) if weights.sum() > 0 else group["score"].mean()
        gewi_v3 = weighted_avg * log_volume

        results.append({
            "year_month": group["year_month"].iloc[0],
            "CD_SETOR": tract_id,
            "GEWI_v1": gewi_v1,
            "GEWI_v2_pos": gewi_pos,
            "GEWI_v2_neg": gewi_neg,
            "GEWI_v3": gewi_v3,
            "tweet_volume": n_total
        })

# Build final DataFrame
df_gewis = pd.DataFrame(results)
df_gewis.to_csv(output_path, index=False)
print(f"\n GEWI indices saved to: {output_path}")

In [ ]:
##  Code for Visualization of GEWI Results

In [ ]:
import pandas as pd

# Path to the CSV file with GEWI scores by census tract
gewis_path = "data/private/gewi/gewis_by_tract.csv"

# Load the data
df = pd.read_csv(gewis_path)

# Display available columns
print(" Available columns in the file:")
print(df.columns.tolist())

# Show a sample of the data
print("\n Sample data preview:")
print(df.head())

In [ ]:
##  Code for Visualization of GEWI v1, v2, and v3 Results (by Month)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Path to the GEWI scores file
path = "data/private/gewi/gewis_by_tract.csv"

# Load the file
df = pd.read_csv(path)

# Group by month and calculate averages
monthly_df = df.groupby("year_month").agg({
    "tweet_volume": "sum",
    "GEWI_v1": "mean",
    "GEWI_v2_pos": "mean",
    "GEWI_v2_neg": "mean",
    "GEWI_v3": "mean"
}).reset_index()

# Style
sns.set(style="whitegrid")

#  Total tweet volume by month
plt.figure(figsize=(14, 5))
sns.lineplot(data=monthly_df, x="year_month", y="tweet_volume", marker="o", linewidth=2)
plt.xticks(rotation=45)
plt.title(" Total Tweet Volume by Month")
plt.ylabel("Number of Tweets")
plt.tight_layout()
plt.show()

#  GEWI v1
plt.figure(figsize=(14, 5))
sns.lineplot(data=monthly_df, x="year_month", y="GEWI_v1", marker="s", linewidth=2)
plt.xticks(rotation=45)
plt.title(" GEWI v1 (Composite Index)")
plt.ylabel("GEWI v1 (average per tract)")
plt.tight_layout()
plt.show()

#  GEWI v2 POS and NEG
plt.figure(figsize=(14, 5))
sns.lineplot(data=monthly_df, x="year_month", y="GEWI_v2_pos", label="Positive", marker="^")
sns.lineplot(data=monthly_df, x="year_month", y="GEWI_v2_neg", label="Negative", marker="v")
plt.xticks(rotation=45)
plt.title(" GEWI v2 — Polarity Decomposition")
plt.ylabel("GEWI v2 Average (per tract)")
plt.legend()
plt.tight_layout()
plt.show()

#  GEWI v3
plt.figure(figsize=(14, 5))
sns.lineplot(data=monthly_df, x="year_month", y="GEWI_v3", marker="d", color="purple", linewidth=2)
plt.xticks(rotation=45)
plt.title(" GEWI v3 (Weighted by Influence)")
plt.ylabel("GEWI v3 (average per tract)")
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import os

# Example path to inspect a file with census tract mapping
path = "data/private/gewi/mapeado_setores_v2/"
files = sorted([f for f in os.listdir(path) if f.endswith(".csv.gz")])
df = pd.read_csv(os.path.join(path, files[0]))

print(" Available columns:")
print(df.columns.tolist())

# Display a sample of relevant columns for GEWI v3
print("\n Sample:")
print(df[["text", "followers"]].head())  # Will raise an error if 'followers' column is missing

In [ ]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm

# Path to census tract-mapped tweet files
input_path = "data/private/gewi/mapeado_setores_v2/"
output_path = "data/private/gewi/"
os.makedirs(output_path, exist_ok=True)

# List files
files = sorted([f for f in os.listdir(input_path) if f.endswith(".csv.gz")])

# List to store results
results = []

print(" Recalculating GEWI v3 with influence weights...")

for file in tqdm(files, desc="Processing files"):
    df = pd.read_csv(os.path.join(input_path, file))

    # Extract year-month from filename
    year_month = file.replace("sector_relevant_messages_", "").replace(".csv.gz", "")

    if df.empty or "score" not in df.columns or "followers" not in df.columns:
        continue

    # Remove rows with missing geolocation or weights
    df = df.dropna(subset=["CD_SETOR", "score", "followers"])

    # Apply influence weight: log(1 + followers)
    df["influence_weight"] = np.log1p(df["followers"].clip(lower=0))  # clip avoids negative log input

    # Calculate weighted GEWI v3 by tract
    grouped = df.groupby("CD_SETOR")
    for tract_id, group in grouped:
        n = len(group)
        weight_sum = group["influence_weight"].sum()

        if weight_sum > 0:
            gewi_v3 = (group["score"] * group["influence_weight"]).sum() / weight_sum * np.log1p(n)
        else:
            gewi_v3 = 0  # or np.nan if preferred

        results.append({
            "year_month": year_month,
            "CD_SETOR": tract_id,
            "GEWI_v3_recalculated": gewi_v3,
            "tweet_volume": n
        })

# Save new CSV with recalculated GEWI v3
df_final = pd.DataFrame(results)
df_final.to_csv(os.path.join(output_path, "gewiv3_recalculated.csv"), index=False)
print(" New GEWI v3 successfully saved.")

In [ ]:
## Visualization of GEWI v3 with Influence Weights

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Path to the file with recalculated GEWI v3
path = "data/private/gewi/gewiv3_recalculated.csv"

# Load the data
df = pd.read_csv(path)
df["year_month"] = df["year_month"].astype(str)

# Sort chronologically
df = df.sort_values("year_month")

# Group by month (average across tracts)
monthly_df = df.groupby("year_month").agg({
    "GEWI_v3_recalculated": "mean",
    "tweet_volume": "sum"
}).reset_index()

#  Visual style
sns.set(style="whitegrid")

#  GEWI v3 with influence weights
plt.figure(figsize=(14, 5))
plt.plot(monthly_df["year_month"], monthly_df["GEWI_v3_recalculated"], marker="o", color="purple")
plt.xticks(rotation=45)
plt.title(" GEWI v3 — Influence-Based (Followers)")
plt.ylabel("Average GEWI v3 per Tract")
plt.xlabel("Year-Month")
plt.grid(True)
plt.tight_layout()
plt.show()

#  Tweet volume per month (for context)
plt.figure(figsize=(14, 4))
plt.plot(monthly_df["year_month"], monthly_df["tweet_volume"], marker="s", linestyle="--", color="gray")
plt.title(" Total Tweet Volume per Month")
plt.ylabel("Number of Tweets")
plt.xlabel("Year-Month")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
#  Final Visualization of GEWIs by Month (Census Tracts)

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Path to the final GEWI file
path = "data/private/gewi/gewis_por_setor.csv"

# Load the dataset
df = pd.read_csv(path)

#  Rename columns for consistency
df = df.rename(columns={
    "GEWI_v1": "gewiv1",
    "GEWI_v2_pos": "gewiv2_pos",
    "GEWI_v2_neg": "gewiv2_neg",
    "GEWI_v3": "gewiv3",
    "volume_tweets": "n_tweets"
})

#  Aggregate by month
monthly_df = df.groupby("ano_mes").agg({
    "n_tweets": "sum",
    "gewiv1": "mean",
    "gewiv2_pos": "mean",
    "gewiv2_neg": "mean",
    "gewiv3": "mean"
}).reset_index()

#  Visual style
sns.set(style="whitegrid")

#  Total tweet volume
plt.figure(figsize=(14, 5))
sns.lineplot(data=monthly_df, x="ano_mes", y="n_tweets", marker="o", linewidth=2)
plt.xticks(rotation=45)
plt.title(" Total Validated Tweets per Month")
plt.ylabel("Total Tweets")
plt.tight_layout()
plt.grid(True)
plt.show()

#  GEWI v1
plt.figure(figsize=(14, 5))
sns.lineplot(data=monthly_df, x="ano_mes", y="gewiv1", marker="o", color="blue")
plt.xticks(rotation=45)
plt.title(" GEWI v1 — Composite Score (Sentiment × Volume)")
plt.ylabel("GEWI v1 (mean)")
plt.tight_layout()
plt.grid(True)
plt.show()

#  GEWI v2 — Positive and Negative
plt.figure(figsize=(14, 5))
sns.lineplot(data=monthly_df, x="ano_mes", y="gewiv2_pos", label="Positive", color="green", marker="o")
sns.lineplot(data=monthly_df, x="ano_mes", y="gewiv2_neg", label="Negative", color="red", marker="o")
plt.xticks(rotation=45)
plt.title(" GEWI v2 — Positive vs Negative Decomposition")
plt.ylabel("GEWI v2 (mean)")
plt.legend()
plt.tight_layout()
plt.grid(True)
plt.show()

#  GEWI v3 — Influence Weighted
plt.figure(figsize=(14, 5))
sns.lineplot(data=monthly_df, x="ano_mes", y="gewiv3", marker="s", color="purple")
plt.xticks(rotation=45)
plt.title(" GEWI v3 — Influence Weighted (log(1 + followers))")
plt.ylabel("GEWI v3 (mean)")
plt.tight_layout()
plt.grid(True)
plt.show()

In [ ]:
## Verification of Influence Weighting in GEWI v3

In [ ]:
import pandas as pd
import numpy as np

# Load one month of data with census tract mapping and GEWI already calculated
path = "data/private/gewi/mapeado_setores_v2/sector_relevant_messages_2020_01.csv.gz"
df = pd.read_csv(path)

# Create the weight column used in GEWI v3
df["weight"] = np.log1p(df["followers"])

# Show basic statistics for the influence weights
print(df["weight"].describe())

# Compare unweighted and weighted sentiment averages
simple_mean = df["score"].mean()
weighted_mean = np.average(df["score"], weights=df["weight"])

print(f"\n Simple sentiment mean: {simple_mean:.4f}")
print(f" Weighted sentiment mean (GEWI v3): {weighted_mean:.4f}")

In [ ]:
## Analysis of the Difference Between GEWI v3 and GEWI v1

In [ ]:
import pandas as pd

# Path to the consolidated GEWI file
path = "data/private/gewi/gewis_por_setor.csv"

# Load the file
df = pd.read_csv(path)

# Check actual column names
print(" Available columns:")
print(df.columns.tolist())

#  Compute the difference (adjust names if needed)
df["difference"] = df["GEWI_v3"] - df["GEWI_v1"]

# Show summary statistics for the difference
print("\n Difference GEWI v3 - v1:")
print(df["difference"].describe())

In [ ]:
#  Block 10 - Normalization of GEWI Indices to [-1, 1] Scale (Per Month)

#This step applies a MinMax normalization to GEWI v1, v2_pos, v2_neg, and v3 scores within each month,
#ensuring comparability across census tracts while preserving temporal structure.

In [ ]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

#  Path to the original GEWI file
input_path = "data/private/gewi/gewis_por_setor.csv"
output_path = "data/private/gewi/gewis_scaled_-1_to_1.csv"

#  Load the original data
df = pd.read_csv(input_path)

#  Columns to normalize
columns_to_scale = ["GEWI_v1", "GEWI_v2_pos", "GEWI_v2_neg", "GEWI_v3"]

#  Group by month and apply MinMaxScaler to each group
scaled_dfs = []
for name, group in df.groupby("ano_mes"):
    scaler = MinMaxScaler(feature_range=(-1, 1))
    scaled_values = scaler.fit_transform(group[columns_to_scale])

    scaled_group = group.copy()
    for idx, col in enumerate(columns_to_scale):
        scaled_group[f"{col}_scaled"] = scaled_values[:, idx]

    scaled_dfs.append(scaled_group)

#  Concatenate all scaled groups
df_scaled = pd.concat(scaled_dfs, ignore_index=True)

#  Save the final DataFrame with scaled columns
df_scaled.to_csv(output_path, index=False)
print(f" Scaled GEWI values saved to: {output_path}")

In [ ]:
##Validation

In [ ]:
import pandas as pd

# Path to the scaled GEWI file
path = "data/private/gewi/gewis_scaled_-1_to_1.csv"

# Load the data
df = pd.read_csv(path)

# Columns of interest
columns_to_display = [
    "ano_mes", "CD_SETOR", "GEWI_v1_scaled", "GEWI_v2_pos_scaled", "GEWI_v2_neg_scaled", "GEWI_v3_scaled"
]

# Show first 10 rows with scaled GEWI values
print(" Sample of GEWI indices scaled to [-1, 1]:\n")
print(df[columns_to_display].head(10))

In [ ]:
##  Block 11 - GEWI Normalization with Quality Flags per Month

In [ ]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# Load original GEWI file
input_path = "data/private/gewi/gewis_por_setor.csv"
output_path = "data/private/gewi/gewis_scaled_flagged.csv"

df = pd.read_csv(input_path)
columns_to_scale = ["GEWI_v1", "GEWI_v2_pos", "GEWI_v2_neg", "GEWI_v3"]

flagged_data = []

for month, group in df.groupby("ano_mes"):
    scaled_group = group.copy()

    # Flag: unique row in month
    if len(group) == 1:
        for col in columns_to_scale:
            scaled_group[f"{col}_scaled"] = 0
        scaled_group["scaling_flag"] = "unique"

    else:
        scaler = MinMaxScaler(feature_range=(-1, 1))
        scaled_values = scaler.fit_transform(group[columns_to_scale])
        for idx, col in enumerate(columns_to_scale):
            scaled_group[f"{col}_scaled"] = scaled_values[:, idx]
        scaled_group["scaling_flag"] = "scaled"

    flagged_data.append(scaled_group)

# Final output
df_final = pd.concat(flagged_data, ignore_index=True)
df_final.to_csv(output_path, index=False)
print(f" GEWI scaled and flagged values saved to: {output_path}")

In [ ]:
## Validation

In [ ]:
import pandas as pd

# Path to the flagged and scaled GEWI file
path = "data/private/gewi/gewis_scaled_flagged.csv"

# Load the file
df = pd.read_csv(path)

# Select columns for validation
columns = [
    "ano_mes", "CD_SETOR",
    "GEWI_v1", "GEWI_v1_scaled",
    "GEWI_v2_pos", "GEWI_v2_pos_scaled",
    "GEWI_v2_neg", "GEWI_v2_neg_scaled",
    "GEWI_v3", "GEWI_v3_scaled",
    "scaling_flag"
]

# Show 10 random rows
sample = df[columns].sample(n=10, random_state=42)
print(" GEWI Validation Sample (original vs scaled):")
print(sample)

In [ ]:
##  Block 12 - Consolidated GEWI per Census Tract (Static Long-Term Index)

In [ ]:
import pandas as pd

# Load the full monthly GEWI file
path = "data/private/gewi/gewis_por_setor.csv"
df = pd.read_csv(path)

# Group by census tract (CD_SETOR) and compute mean across time
df_static = df.groupby("CD_SETOR").agg({
    "GEWI_v1": "mean",
    "GEWI_v2_pos": "mean",
    "GEWI_v2_neg": "mean",
    "GEWI_v3": "mean",
    "volume_tweets": "sum",  # Optional: total activity
}).reset_index()

# Save to file
output_path = "data/private/gewi/gewis_static_by_tract.csv"
df_static.to_csv(output_path, index=False)
print(f" Static GEWI by tract saved to: {output_path}")

In [ ]:
## Quality check

In [ ]:
import pandas as pd

# Path to the consolidated file with average GEWI per census tract
path = "data/private/gewi/gewis_static_by_tract.csv"

# Load the file
df_static = pd.read_csv(path)

# Display the first few rows
print(" Sample of consolidated GEWI values by census tract:")
print(df_static.head(10))  # Displays the first 10 rows; you can adjust the number

In [ ]:
## Check - difference between GEWI_v1 e GEWI_v3

In [ ]:
import pandas as pd
import numpy as np

# Caminho do arquivo consolidado com os GEWIs
path = "data/private/gewi/gewis_static_by_tract.csv"
df = pd.read_csv(path)

# Exibe as colunas para confirmar a presença dos GEWIs
print(" Colunas disponíveis no arquivo consolidado:", df.columns.tolist())

# Verifica as primeiras linhas do DataFrame para observar os valores
print(" Amostra dos dados:")
print(df.head())

# Calcular a diferença entre GEWI_v1 e GEWI_v3, para verificar inconsistências
df['diferenca_GEWI'] = df['GEWI_v1'] - df['GEWI_v3']

# Exibe estatísticas básicas da diferença entre GEWI_v1 e GEWI_v3
print("\n Estatísticas da diferença GEWI_v1 - GEWI_v3:")
print(df['diferenca_GEWI'].describe())

# Salvar a comparação para referência futura (opcional)
output_path = "data/private/gewi/gewis_comparacao.csv"
df.to_csv(output_path, index=False)
print(f" Comparação dos GEWIs salva em: {output_path}")

In [ ]:
#The difference between GEWI_v1 and GEWI_v3 is minimal, with a mean difference close to zero and a small standard deviation, indicating they are nearly identical.